# Table 8: Wildcard prevalence in source domains
Recreate Table 8 in *Dissecting NSEC and NSEC3 Deployment in the Wild* from the April 6, 2026 summary files. No measurement or wildcard-classification scripts are rerun.

Inputs:
- `script/res/wildcard/20260406_wildcard_stats.csv`: wildcard denial, acknowledgement, and candidate counts.
- `script/res/20260406_domcnt_stats.csv`: source NSEC/NSEC3 domain totals, including auxiliary scanning.

For each dataset and mechanism:

`ACK wildcard % = 100 × wildcard acknowledgement count / mechanism domain total`

`Candidates % = 100 × wildcard candidate count / mechanism domain total`


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATE = "20260406"
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "script/res/wildcard" / f"{DATE}_wildcard_stats.csv").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside dnsnsec26 or set REPO_ROOT explicitly.")
RES_DIR = REPO_ROOT / "script/res"
OUTPUT_DIR = REPO_ROOT / "notebook/results"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org", "umbrella": "Umbrella",
          "tranco": "Tranco", "radar": "Radar", "majestic": "Majestic"}

## Load counts and denominators
Join by dataset name rather than file row position. Validate complete coverage, integer counts, positive totals, and counts that do not exceed each mechanism's domain total. Acknowledgement, denial, and candidate columns are not assumed to form mutually exclusive categories.

In [3]:
wildcard_counts = pd.read_csv(RES_DIR / "wildcard" / f"{DATE}_wildcard_stats.csv")
domain_counts = pd.read_csv(RES_DIR / f"{DATE}_domcnt_stats.csv")
for frame, label in [(wildcard_counts, "wildcard"), (domain_counts, "domain")]:
    if frame["zone"].duplicated().any() or set(frame["zone"]) != set(ZONES):
        raise ValueError(f"{label}: expected one row for every dataset.")
wildcard_counts = wildcard_counts.set_index("zone").loc[ZONES]
domain_counts = domain_counts.set_index("zone").loc[ZONES]

frames = []
for mechanism in ("nsec", "nsec3"):
    frame = pd.DataFrame({
        "total_domains": domain_counts[mechanism],
        "deny_wildcard": wildcard_counts[f"{mechanism}_wildcard_deny"],
        "ack_wildcard": wildcard_counts[f"{mechanism}_wildcard_ack"],
        "candidates": wildcard_counts[f"{mechanism}_wildcard_toquery"],
    })
    for column in frame.columns:
        values = pd.to_numeric(frame[column], errors="raise")
        if values.isna().any() or values.lt(0).any() or (values % 1 != 0).any():
            raise ValueError(f"{mechanism}: {column} must contain nonnegative integers.")
        frame[column] = values.astype("int64")
    if frame["total_domains"].le(0).any():
        raise ValueError("Domain denominators must be positive.")
    if frame[["deny_wildcard", "ack_wildcard", "candidates"]].gt(frame["total_domains"], axis=0).any().any():
        raise ValueError("A wildcard count exceeds its mechanism domain total.")
    frame["mechanism"] = mechanism.upper()
    frames.append(frame.reset_index())
stats = pd.concat(frames, ignore_index=True)
stats["ack_pct"] = 100 * stats["ack_wildcard"] / stats["total_domains"]
stats["candidates_pct"] = 100 * stats["candidates"] / stats["total_domains"]

## Recreated Table 8
For each mechanism, display Deny wildcard, ACK wildcard, ACK percentage, Candidates, and Candidate percentage in the paper's order. Integer counts are preserved; percentage rounding applies only to presentation.

In [4]:
fields = ["deny_wildcard", "ack_wildcard", "ack_pct", "candidates", "candidates_pct"]
labels = ["Deny wildcard", "ACK wildcard", "ACK %", "Candidates", "Candidates %"]
parts = []
for mechanism in ("NSEC", "NSEC3"):
    part = stats.loc[stats["mechanism"].eq(mechanism)].set_index("zone").loc[ZONES, fields].copy()
    part.columns = pd.MultiIndex.from_tuples([(mechanism, label) for label in labels],
                                             names=["Mechanism", "Statistic"])
    parts.append(part)
table8 = pd.concat(parts, axis=1)
table8.index = table8.index.map(LABELS)
table8.index.name = "Dataset"
formatted_table8 = table8.copy().astype(object)
for column in table8.columns:
    if column[1].endswith("%"):
        formatted_table8[column] = table8[column].map(lambda value: f"{value:.2f}%")
    else:
        formatted_table8[column] = table8[column].map(lambda value: f"{value:,d}")
display(formatted_table8)

Mechanism          NSEC                                               \
Statistic Deny wildcard ACK wildcard   ACK % Candidates Candidates %   
Dataset                                                                
.com              1,878      227,716  14.44%    325,909       20.67%   
.net                212       18,642  11.61%     33,940       21.14%   
.org                202       17,723  12.11%     28,985       19.81%   
Umbrella             40          671   4.13%      4,554       28.02%   
Tranco               81        1,883   3.84%     10,232       20.85%   
Radar               111        2,791   4.29%     14,686       22.58%   
Majestic             72        2,256   5.50%      8,722       21.28%   

Mechanism         NSEC3                                               
Statistic Deny wildcard ACK wildcard   ACK % Candidates Candidates %  
Dataset                                                               
.com              2,698      453,649   6.75%    729,193       10.85%  
.net                247       39,005   7.18%     63,230       11.64%  
.org                169       39,130   7.03%     58,701       10.55%  
Umbrella             16        1,157  12.15%      1,666       17.49%  
Tranco               16        5,250  17.44%      6,866       22.80%  
Radar                15        7,241  19.98%      9,598       26.48%  
Majestic              4        5,886  17.16%      7,835       22.85%

## Export
- `table8_stats.csv`: counts, mechanism denominators, and unrounded percentages.
- `table8_formatted.csv`: the paper's table layout with two header rows.
- `table8_paper_differences.csv`: calculated minus published values at the paper's precision; percentage differences are percentage points.

Rerunning this cell replaces these exports.

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stats.to_csv(OUTPUT_DIR / "table8_stats.csv", index=False)
print(f"Saved Table 8 results to {OUTPUT_DIR}")

Saved Table 8 results to /u/yz6me/dnsnsec26/notebook/results
